In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/sample_submission.csv
/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/train.csv
/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/test.csv


In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import category_encoders as ce
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier,VotingClassifier,StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [3]:
df=pd.read_csv('/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/train.csv')
print(df.info())
df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279746 entries, 0 to 279745
Data columns (total 60 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   space_id                      279746 non-null  int64  
 1   operator_ref                  279142 non-null  object 
 2   operator_platform_age_years   279142 non-null  float64
 3   operator_platform_age_months  279142 non-null  float64
 4   operator_active_years         279140 non-null  float64
 5   operator_active_months        279140 non-null  float64
 6   operator_class                279746 non-null  object 
 7   operator_certified            279142 non-null  object 
 8   operator_portfolio_size       279142 non-null  float64
 9   operator_has_avatar           279142 non-null  object 
 10  operator_id_verified          279142 non-null  object 
 11  zone                          279746 non-null  object 
 12  borough                       65295 non-null

,space_id,operator_ref,operator_platform_age_years,operator_platform_age_months,operator_active_years,operator_active_months,operator_class,operator_certified,operator_portfolio_size,operator_has_avatar,...,score_access,score_value,compliance_status,operator_metro_units,operator_metro_whole,operator_metro_private,operator_metro_shared,reviews_per_month,metro,price_tier
0,0,Bralchu,12.0,8.0,7.0,8.0,independent_operator,f,2.0,t,...,9.12,9.54,unlicensed,2,0,2,0,0.42,MTR-09,2
1,1,Grorfian,2.0,2.0,2.0,2.0,independent_operator,f,3.0,t,...,NaN,NaN,unlicensed,3,1,2,0,NaN,MTR-01,1
2,2,Snelenr,10.0,10.0,7.0,10.0,independent_operator,f,1.0,t,...,10.00,10.00,unlicensed,1,1,0,0,0.03,MTR-01,2
3,3,Spesspe,2.0,1.0,2.0,1.0,unverified_operator,t,1.0,t,...,10.00,10.00,unlicensed,1,1,0,0,0.60,MTR-09,4
4,4,Gosho,2.0,7.0,2.0,4.0,unverified_operator,t,1.0,t,...,9.42,9.74,licensed,1,1,0,0,4.64,MTR-08,3
5,5,Harthynrn,13.0,7.0,13.0,6.0,independent_operator,f,1.0,t,...,9.84,9.38,licensed,1,1,0,0,0.79,MTR-11,3
6,6,Vianchessl,10.0,3.0,10.0,3.0,managed_operator,t,232.0,t,...,10.00,9.56,unlicensed,148,148,0,0,0.23,MTR-08,1
7,7,Sordessx,12.0,11.0,7.0,3.0,independent_operator,t,5.0,t,...,9.90,9.86,exempt,5,5,0,0,3.29,MTR-03,4
8,8,Praless,13.0,7.0,13.0,7.0,independent_operator,t,5.0,t,...,10.00,9.00,exempt,5,5,0,0,0.13,MTR-11,2
9,9,Krafyn,2.0,3.0,2.0,3.0,unverified_operator,f,12.0,t,...,9.72,8.72,exempt,12,5,7,0,1.98,MTR-13,1


In [4]:
df.isna().sum()

space_id                             0
operator_ref                       604
operator_platform_age_years        604
operator_platform_age_months       604
operator_active_years              606
operator_active_months             606
operator_class                       0
operator_certified                 604
operator_portfolio_size            604
operator_has_avatar                604
operator_id_verified               604
zone                                 0
borough                         214451
geo_lat                              0
geo_lon                              0
space_class                          0
access_type                          0
max_seats                            0
utility_rooms                    29899
zones                            48414
workstations                     18783
facilities                           0
snapshot_start_date                  0
snapshot_end_date                    0
min_term_days                       41
max_term_days            

# 1. Exploratory Data Analysis (EDA)

In [5]:
#NUMERICAL
score_cols =['score_overall', 'score_accuracy', 'score_condition','score_onboarding', 'score_support', 'score_access', 'score_value', 'reviews_per_month']
df[score_cols] =df[score_cols].fillna(0) # 0=No review

df['borough'] =df['borough'].fillna('Unknown')

numeric_cols =df.select_dtypes(include=['float64','int64']).columns.drop(score_cols,errors='ignore')
for col in numeric_cols:
    df[col] =df[col].fillna(df[col].median())

#CATEGORICAL 
cat_cols =df.select_dtypes(include=['object']).columns.drop(['borough'])
for col in cat_cols:
    df[col] =df[col].fillna(df[col].mode()[0])


df.isnull().sum()

space_id                        0
operator_ref                    0
operator_platform_age_years     0
operator_platform_age_months    0
operator_active_years           0
operator_active_months          0
operator_class                  0
operator_certified              0
operator_portfolio_size         0
operator_has_avatar             0
operator_id_verified            0
zone                            0
borough                         0
geo_lat                         0
geo_lon                         0
space_class                     0
access_type                     0
max_seats                       0
utility_rooms                   0
zones                           0
workstations                    0
facilities                      0
snapshot_start_date             0
snapshot_end_date               0
min_term_days                   0
max_term_days                   0
min_term_floor                  0
min_term_ceiling                0
max_term_floor                  0
max_term_ceili

In [6]:
df['facility_count'] =df['facilities'].apply(lambda x:len(str(x).split(',')))

date_cols = ['snapshot_start_date', 'snapshot_end_date', 'first_review_date', 'last_review_date']
for col in date_cols:
    df[col] =pd.to_datetime(df[col], errors='coerce')

df['days_since_first_review'] =(df['snapshot_start_date'] - df['first_review_date']).dt.days
df['days_since_first_review'] =df['days_since_first_review'].fillna(0)

cols_to_drop =['facilities', 'snapshot_start_date', 'snapshot_end_date','first_review_date', 'last_review_date', 'operator_ref', 'space_id']
df =df.drop(columns=cols_to_drop,errors='ignore')

print(df.shape)
df.head(10)

(279746, 55)


,operator_platform_age_years,operator_platform_age_months,operator_active_years,operator_active_months,operator_class,operator_certified,operator_portfolio_size,operator_has_avatar,operator_id_verified,zone,...,compliance_status,operator_metro_units,operator_metro_whole,operator_metro_private,operator_metro_shared,reviews_per_month,metro,price_tier,facility_count,days_since_first_review
0,12.0,8.0,7.0,8.0,independent_operator,f,2.0,t,t,ZN-0557,...,unlicensed,2,0,2,0,0.42,MTR-09,2,40,2847
1,2.0,2.0,2.0,2.0,independent_operator,f,3.0,t,t,ZN-0082,...,unlicensed,3,1,2,0,0.00,MTR-01,1,42,55
2,10.0,10.0,7.0,10.0,independent_operator,f,1.0,t,t,ZN-0082,...,unlicensed,1,1,0,0,0.03,MTR-01,2,43,1803
3,2.0,1.0,2.0,1.0,unverified_operator,t,1.0,t,t,ZN-0654,...,unlicensed,1,1,0,0,0.60,MTR-09,4,40,82
4,2.0,7.0,2.0,4.0,unverified_operator,t,1.0,t,t,ZN-0387,...,licensed,1,1,0,0,4.64,MTR-08,3,34,804
5,13.0,7.0,13.0,6.0,independent_operator,f,1.0,t,t,ZN-0871,...,licensed,1,1,0,0,0.79,MTR-11,3,38,4921
6,10.0,3.0,10.0,3.0,managed_operator,t,232.0,t,t,ZN-1080,...,unlicensed,148,148,0,0,0.23,MTR-08,1,26,3743
7,12.0,11.0,7.0,3.0,independent_operator,t,5.0,t,t,ZN-0663,...,exempt,5,5,0,0,3.29,MTR-03,4,18,2374
8,13.0,7.0,13.0,7.0,independent_operator,t,5.0,t,t,ZN-1349,...,exempt,5,5,0,0,0.13,MTR-11,2,29,958
9,2.0,3.0,2.0,3.0,unverified_operator,f,12.0,t,t,ZN-0323,...,exempt,12,5,7,0,1.98,MTR-13,1,26,728


# 2. Data Preprocessing & Cleaning

In [7]:
X =df.drop(columns=['price_tier'])
y =df['price_tier']

numeric_cols =X.select_dtypes(include=['int64', 'float64']).columns.tolist()
te_cols =['zone', 'space_class','borough'] 
ohe_cols = ['operator_class', 'operator_certified', 'operator_has_avatar','operator_id_verified', 'access_type', 'bookable','compliance_status', 'metro']

preprocessor =ColumnTransformer(
    transformers=[
        ('scale',StandardScaler(), numeric_cols),
        ('encode_ohe',OneHotEncoder(handle_unknown='ignore', sparse_output=False), ohe_cols),
        ('encode_te',ce.TargetEncoder(smoothing=10), te_cols)
    ], remainder='drop')

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=7, stratify=y
)

print("Training size :",X_train.shape)
print("Testing size :",X_val.shape)

X_train_processed =preprocessor.fit_transform(X_train, y_train)
X_val_processed =preprocessor.transform(X_val)

Training size : (223796, 54)
Testing size : (55950, 54)


In [9]:
#CLEANING TEST DATA : 
test_df = pd.read_csv('/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/test.csv')
sample = pd.read_csv('/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/sample_submission.csv')
test_ids = test_df['space_id']
test_df['facility_count'] = test_df['facilities'].apply(lambda x: len(str(x).split(',')))

for col in date_cols:
    test_df[col] = pd.to_datetime(test_df[col], errors='coerce')

test_df['days_since_first_review'] = (test_df['snapshot_start_date'] - test_df['first_review_date']).dt.days
test_df['days_since_first_review'] = test_df['days_since_first_review'].fillna(0)

test_df[score_cols] = test_df[score_cols].fillna(0)
test_df['borough'] = test_df['borough'].fillna('Unknown')

for col in numeric_cols:
    test_df[col] = test_df[col].fillna(test_df[col].median())

for col in cat_cols:
    test_df[col] = test_df[col].fillna(test_df[col].mode()[0])
test_df = test_df.drop(columns=cols_to_drop, errors='ignore')
X_test_processed = preprocessor.transform(test_df)

# 3. Model Training & Hyperparameter Tuning

In [10]:
#Logistic Regression
log_reg =LogisticRegression(max_iter=1000, class_weight='balanced', random_state=7)
log_reg.fit(X_train_processed, y_train)

val_predictions =log_reg.predict(X_val_processed)
baseline_score =f1_score(y_val, val_predictions, average='macro')

print("Logistic Regression Macro F1 Score: ",baseline_score)
test_predictions = log_reg.predict(X_test_processed)
submission = pd.DataFrame({
    'ID': test_ids,
    'price_tier': test_predictions
})
submission.to_csv('submission.csv', index=False)

Logistic Regression Macro F1 Score:  0.638759424222948


In [11]:
#Decision Tree
param_grid_dt ={
    'max_depth':[5, 10, 15, 20, None], 
    'min_samples_split':[2, 10, 50, 100]
}

base_dt =DecisionTreeClassifier(class_weight='balanced', random_state=7)
grid_search =GridSearchCV(
    estimator=base_dt, 
    param_grid=param_grid_dt, 
    cv=3, 
    scoring='f1_macro', 
    n_jobs=-1, 
    verbose=1
)
grid_search.fit(X_train_processed, y_train)

print("Best parameters found:" ,grid_search.best_params_)

best_dt_model =grid_search.best_estimator_
dt_val_preds = best_dt_model.predict(X_val_processed)
dt_score =f1_score(y_val, dt_val_preds, average='macro')
print(f"Tuned Decision Tree Validation Score: ",dt_score)

dt_test_preds =best_dt_model.predict(X_test_processed)
submission_tuned_dt = pd.DataFrame({
    'ID':test_ids,
    'price_tier':dt_test_preds
})
submission_tuned_dt.to_csv('submission.csv', index=False)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters found: {'max_depth': 15, 'min_samples_split': 100}
Tuned Decision Tree Validation Score:  0.6492384525300608


In [12]:
#Random Forest 
rf_model =RandomForestClassifier(
    n_estimators=100, 
    max_depth=15, 
    min_samples_split=100, 
    class_weight='balanced', 
    n_jobs=-1, 
    random_state=7
)
rf_model.fit(X_train_processed, y_train)

rf_val_preds= rf_model.predict(X_val_processed)
rf_score =f1_score(y_val, rf_val_preds, average='macro')
print("Random Forest Validation Score: ",rf_score)

rf_test_preds = rf_model.predict(X_test_processed)
sample['price_tier'] = rf_test_preds
sample.to_csv('submission.csv', index=False)

Random Forest Validation Score:  0.6814995630862799


In [13]:
#XGBoost 
y_train_xgb=y_train-1

xgb_model =XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    n_jobs=-1,
    random_state=7
)
xgb_model.fit(X_train_processed, y_train_xgb)

xgb_val_preds =xgb_model.predict(X_val_processed)+1
xgb_score = f1_score(y_val, xgb_val_preds, average='macro')
print("XGBoost Validation Score: ",xgb_score)

xgb_test_preds = xgb_model.predict(X_test_processed) + 1

sample['price_tier'] =xgb_test_preds
sample.to_csv('submission.csv', index=False)

XGBoost Validation Score:  0.6771793701739526


In [14]:
#LightGBM
y_train_lgbm= y_train-1

lgbm_model = LGBMClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    class_weight='balanced',
    random_state=7,
    n_jobs=-1,
    verbose=-1 
)
lgbm_model.fit(X_train_processed, y_train_lgbm)

lgbm_val_preds=lgbm_model.predict(X_val_processed) + 1
lgbm_score =f1_score(y_val, lgbm_val_preds, average='macro')
print("LightGBM Validation Score: ",lgbm_score)

lgbm_test_preds =lgbm_model.predict(X_test_processed) + 1

sample['price_tier']=lgbm_test_preds
sample.to_csv('submission.csv', index=False)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Validation Score:  0.6848371154612215


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [15]:
#CatBoost
y_train_cat= y_train - 1

cat_model=CatBoostClassifier(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    auto_class_weights='Balanced',
    random_seed=7,
    verbose=0 
)
cat_model.fit(X_train_processed, y_train_cat)

cat_val_preds= cat_model.predict(X_val_processed).flatten() + 1
cat_score =f1_score(y_val, cat_val_preds, average='macro')
print("CatBoost Validation Score: ",cat_score)

cat_test_preds =cat_model.predict(X_test_processed).flatten() + 1

sample['price_tier']=cat_test_preds
sample.to_csv('submission.csv', index=False)

CatBoost Validation Score:  0.6635034087486098


In [16]:
#Voting Classifier (Soft Voting + Weighted Voting)
y_train_shifted = y_train - 1

lgbm = LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, class_weight='balanced', random_state=7, n_jobs=-1, verbose=-1)
xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=7, n_jobs=-1)
rf = RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_split=100, class_weight='balanced', random_state=7, n_jobs=-1)

voting_model =VotingClassifier(
    estimators=[('lgbm', lgbm), ('xgb', xgb), ('rf', rf)],
    voting='soft',
    weights=[3, 2, 1] 
)
voting_model.fit(X_train_processed, y_train_shifted)

voting_val_preds =voting_model.predict(X_val_processed) + 1
voting_score =f1_score(y_val, voting_val_preds, average='macro')
print("Tweaked Voting Ensemble Validation Score: ",voting_score)

voting_test_preds = voting_model.predict(X_test_processed) + 1

sample['price_tier'] = voting_test_preds
sample.to_csv('submission.csv', index=False)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Tweaked Voting Ensemble Validation Score:  0.6864310683601844


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [17]:
#Stacking Ensemble
y_train_shifted = y_train - 1

lgbm = LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, class_weight='balanced', random_state=7, n_jobs=-1, verbose=-1)
xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=7, n_jobs=-1)
rf = RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_split=100, class_weight='balanced', random_state=7, n_jobs=-1)

meta_learner = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=7)

stacking_model = StackingClassifier(
    estimators=[('lgbm', lgbm), ('xgb', xgb), ('rf', rf)],
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1
)
stacking_model.fit(X_train_processed, y_train_shifted)

stacking_val_preds =stacking_model.predict(X_val_processed) + 1
stacking_score =f1_score(y_val, stacking_val_preds, average='macro')
print("Stacking Ensemble Validation Score: ",stacking_score)

stacking_test_preds = stacking_model.predict(X_test_processed) + 1

sample['price_tier'] = stacking_test_preds
sample.to_csv('submission.csv', index=False)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

Stacking Ensemble Validation Score:  0.6887392837377433


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
